# Quant Finance – Learning Notebook
# Module 1: Market Fundamentals & Return Calculation

**Author:** Florian Ebner  
**Purpose:** Structured self-study for Quantitative Risk / Quantitative Research roles

---

## Source References

- Markowitz, H. (1952). *Portfolio Selection*. Journal of Finance, 7(1), 77–91.
- Sharpe, W.F. (1964). *Capital Asset Prices: A Theory of Market Equilibrium*. Journal of Finance, 19(3), 425–442.
- Hull, J.C. (2018). *Options, Futures, and Other Derivatives* (10th ed.). Pearson.
- Grinold, R. & Kahn, R. (1999). *Active Portfolio Management*. McGraw-Hill.
- Fabozzi, F.J. (2007). *Fixed Income Analysis* (2nd ed.). CFA Institute.
- CFA Institute. (2020). *CFA Program Curriculum*. Level I–III.
- de Prado, M.L. (2018). *Advances in Financial Machine Learning*. Wiley.

---


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats

np.random.seed(42)

plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': '#f8f9fa',
    'axes.grid': True, 'grid.color': '#e0e0e0',
    'font.family': 'sans-serif', 'axes.spines.top': False, 'axes.spines.right': False,
})
print('Setup complete. Module 1 – Market Fundamentals & Return Calculation')

---
## 1. Return Calculation: Daily, Monthly, Annual

### Theory

Return calculation is the absolute starting point of every quantitative analysis. There are two fundamental types:

**Simple Return (arithmetic return):**
$$r_t = \frac{P_t - P_{t-1}}{P_{t-1}} = \frac{P_t}{P_{t-1}} - 1$$

**Log Return (continuously compounded return):**
$$r_t^{\log} = \ln\left(\frac{P_t}{P_{t-1}}\right)$$

### When to use which?

**Log Returns** are the standard in practice because:
1. **Time additivity:** Log returns over T periods sum directly: $r_{1\to T}^{\log} = \sum_{t=1}^T r_t^{\log}$. Simple returns do not have this property.
2. **Normality assumption:** Log returns are approximately normally distributed (strictly: log-normal prices → normally distributed log returns), enabling parametric statistics.
3. **No negative prices:** Since $\ln(P_t/P_{t-1})$ takes real values, prices are implicitly always positive.

**Simple Returns** are preferred for:
- Portfolio returns (weighted sum of simple returns = portfolio simple return)
- Short horizons (difference from log return is negligible for daily returns)

### Annualisation

To make returns of different frequencies comparable, scale to annual basis:

$$r_{\text{ann}} = (1 + r_{\text{daily}})^{252} - 1 \approx r_{\text{daily}} \times 252 \quad (\text{approximation})$$

**Trading days:** In practice, 252 trading days per year are used (not 365 calendar days), since markets are closed on weekends and public holidays.

**Source:** Hull (2018), Ch. 3; CFA Institute Curriculum Level I, Reading 7

In [ ]:
# ── Simulate stock price (Geometric Brownian Motion) ─────────────────────────
T        = 756          # 3 years of trading days
mu_ann   = 0.08
sig_ann  = 0.20
mu_d     = mu_ann  / 252
sig_d    = sig_ann / np.sqrt(252)
S0       = 100.0

log_returns_daily = np.random.normal(mu_d, sig_d, T)
prices = S0 * np.exp(np.cumsum(log_returns_daily))
prices = np.insert(prices, 0, S0)

# ── Return calculation ────────────────────────────────────────────────────────
simple_returns = pd.Series(prices).pct_change().dropna()
log_returns    = pd.Series(np.log(prices[1:] / prices[:-1]))

# ── Monthly returns (resampling) ──────────────────────────────────────────────
dates      = pd.bdate_range('2021-01-01', periods=T+1)
price_ser  = pd.Series(prices, index=dates)
monthly_r  = price_ser.resample('ME').last().pct_change().dropna()
yearly_r   = price_ser.resample('YE').last().pct_change().dropna()

# ── Annualisation ─────────────────────────────────────────────────────────────
cagr    = (prices[-1] / prices[0]) ** (252 / T) - 1
ann_vol = simple_returns.std() * np.sqrt(252)

print(f'Total Return (Simple)        : {(prices[-1]/prices[0]-1)*100:.2f}%')
print(f'CAGR (annualised)            : {cagr*100:.2f}%')
print(f'Annualised Volatility        : {ann_vol*100:.2f}%')
print(f'Mean daily return            : {simple_returns.mean()*100:.4f}%')
print(f'Diff Simple vs Log Return    : {(simple_returns.mean()-log_returns.mean())*10000:.4f} bps')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(dates, prices, color='#4a90d9', lw=1.5)
axes[0].set_title('Simulated Stock Price', fontweight='bold')
axes[0].set_ylabel('Price (€)')

axes[1].hist(simple_returns * 100, bins=50, color='#4a90d9', alpha=0.7, edgecolor='white')
axes[1].axvline(0, color='red', lw=1, ls='--')
axes[1].set_title('Distribution of Daily Returns (%)', fontweight='bold')
axes[1].set_xlabel('Daily Return (%)')

axes[2].bar(range(len(yearly_r)), yearly_r * 100,
            color=['#2ecc71' if r > 0 else '#e74c3c' for r in yearly_r], edgecolor='white')
axes[2].set_xticks(range(len(yearly_r)))
axes[2].set_xticklabels([str(y.year) for y in yearly_r.index])
axes[2].set_title('Annual Returns (%)', fontweight='bold')
axes[2].set_ylabel('Return (%)')
axes[2].axhline(0, color='black', lw=0.8)

plt.tight_layout()
plt.savefig('returns_overview.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 2. Volatility

### Theory

Volatility is the **most important risk measure** in finance. It measures the dispersion of returns around their expected value.

**Historical (Realised) Volatility:**
$$\sigma = \sqrt{\frac{1}{T-1} \sum_{t=1}^T (r_t - \bar{r})^2}$$

**Annualisation:** $\sigma_{\text{ann}} = \sigma_{\text{daily}} \times \sqrt{252}$ (square-root-of-time rule)

### Implied Volatility vs. Historical Volatility

- **Historical Volatility:** Backward-looking, computed from past prices. Measures *what was*.
- **Implied Volatility (IV):** Back-computed from option prices via Black-Scholes. Measures *what the market expects*. The **VIX** is the most well-known IV index (S&P 500, 30-day horizon).

### Volatility Clustering (ARCH/GARCH)

Empirically observed: high volatility follows high volatility, low follows low. These **volatility clusters** are modelled by **GARCH models** (Generalized AutoRegressive Conditional Heteroskedasticity):

$$\sigma_t^2 = \omega + \alpha \epsilon_{t-1}^2 + \beta \sigma_{t-1}^2$$

A simple GARCH(1,1): today's volatility depends on yesterday's shock ($\epsilon_{t-1}^2$) and yesterday's volatility ($\sigma_{t-1}^2$). Typical parameters: $\alpha \approx 0.1$, $\beta \approx 0.85$.

### Fat Tails and Kurtosis

Real return distributions have **fatter tails** than the normal distribution — extreme losses occur more frequently than a normal model predicts. Measured by **Excess Kurtosis** (Kurtosis - 3): Normal distribution = 0, real equities typically 2–10.

**Source:** Engle, R. (1982). *Autoregressive Conditional Heteroscedasticity*. Econometrica, 50(4); Hull (2018), Ch. 23

In [ ]:
# ── Rolling Volatility ───────────────────────────────────────────────────────
roll_vol_21  = simple_returns.rolling(21).std()  * np.sqrt(252) * 100
roll_vol_63  = simple_returns.rolling(63).std()  * np.sqrt(252) * 100
roll_vol_252 = simple_returns.rolling(252).std() * np.sqrt(252) * 100

# ── Kurtosis and Skewness ─────────────────────────────────────────────────────
excess_kurtosis = simple_returns.kurtosis()
skewness        = simple_returns.skew()

print(f'Annualised Volatility (252-day): {roll_vol_252.dropna().iloc[-1]:.2f}%')
print(f'Excess Kurtosis                : {excess_kurtosis:.4f}  (Normal = 0)')
print(f'Skewness                       : {skewness:.4f}  (Normal = 0)')

jb_stat, jb_p = stats.jarque_bera(simple_returns)
print(f'Jarque-Bera Test: stat={jb_stat:.2f}, p={jb_p:.6f} →',
      'Not normally distributed ✓' if jb_p < 0.05 else 'Normality not rejected')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(dates[1:], roll_vol_21,  color='#e74c3c', lw=1,   alpha=0.8, label='21-day')
axes[0].plot(dates[1:], roll_vol_63,  color='#e67e22', lw=1.5, alpha=0.8, label='63-day')
axes[0].plot(dates[1:], roll_vol_252, color='#2c3e50', lw=2,   alpha=0.9, label='252-day')
axes[0].set_title('Rolling Annualised Volatility', fontweight='bold')
axes[0].set_ylabel('Volatility (% p.a.)')
axes[0].legend()

x = np.linspace(simple_returns.min(), simple_returns.max(), 300)
axes[1].hist(simple_returns, bins=60, density=True, color='#4a90d9',
             alpha=0.6, edgecolor='white', label='Empirical')
axes[1].plot(x, stats.norm.pdf(x, simple_returns.mean(), simple_returns.std()),
             'r-', lw=2, label='Normal distribution')
axes[1].set_title(f'Fat Tails: Excess Kurtosis={excess_kurtosis:.2f}', fontweight='bold')
axes[1].legend()

stats.probplot(simple_returns, dist='norm', plot=axes[2])
axes[2].set_title('QQ-Plot (Deviation from Normality)', fontweight='bold')
axes[2].get_lines()[0].set(color='#4a90d9', markersize=2)

plt.tight_layout()
plt.savefig('volatility.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 3. Correlation

### Theory

**Pearson Correlation** measures the linear dependence between two time series:
$$\rho_{XY} = \frac{\text{Cov}(X,Y)}{\sigma_X \cdot \sigma_Y} \in [-1, +1]$$

**Portfolio interpretation:**
- $\rho = +1$: Perfectly positively correlated — no diversification possible
- $\rho = 0$: No linear relationship — maximum diversification
- $\rho = -1$: Perfectly negatively correlated — full hedging possible

### Correlation Instability and Correlation Breakdown

The most dangerous phenomenon in risk modelling: during crises, **correlations between equities spike sharply**. Assets that are normally weakly related (ρ ≈ 0.3) can fall together in a crash (ρ ≈ 0.9). This makes diversification ineffective precisely when it is needed most.

Documented in: Longin & Solnik (2001), *Extreme Correlation of International Equity Markets*; also observed in the 2008 financial crisis.

### Rolling Correlation

Since correlations are time-varying, **rolling correlation** (computed over a sliding window) is far more informative than a static estimate over the entire sample.

### Rank Correlation (Spearman)

**Spearman correlation** measures monotone (not necessarily linear) relationships and is more robust to outliers. Frequently used in factor research.

**Source:** Longin, F. & Solnik, B. (2001). Journal of Finance, 56(2), 649–676. Hull (2018), Ch. 9

In [ ]:
# ── Two asset return series ───────────────────────────────────────────────────
r1 = pd.Series(np.random.normal(0.0004, 0.012, T))
noise = np.random.normal(0, 0.009, T)
r2    = pd.Series(0.6 * r1 + np.sqrt(1 - 0.6**2) * noise)

pearson_r,  pearson_p  = stats.pearsonr(r1, r2)
spearman_r, spearman_p = stats.spearmanr(r1, r2)

print(f'Pearson Correlation  : {pearson_r:.4f}  (p={pearson_p:.2e})')
print(f'Spearman Correlation : {spearman_r:.4f}  (p={spearman_p:.2e})')

roll_corr_63  = r1.rolling(63).corr(r2)
roll_corr_126 = r1.rolling(126).corr(r2)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(r1 * 100, r2 * 100, alpha=0.3, s=8, color='#4a90d9')
m, b = np.polyfit(r1, r2, 1)
x_fit = np.linspace(r1.min(), r1.max(), 100)
axes[0].plot(x_fit * 100, (m * x_fit + b) * 100, 'r-', lw=2)
axes[0].set_xlabel('Asset 1 Daily Return (%)')
axes[0].set_ylabel('Asset 2 Daily Return (%)')
axes[0].set_title(f'Scatter Plot: ρ = {pearson_r:.3f}', fontweight='bold')

axes[1].plot(roll_corr_63,  color='#e74c3c', lw=1.2, alpha=0.8, label='63-day (3M)')
axes[1].plot(roll_corr_126, color='#2c3e50', lw=2,   alpha=0.9, label='126-day (6M)')
axes[1].axhline(pearson_r, color='grey', ls='--', lw=1, label=f'Full sample ρ={pearson_r:.3f}')
axes[1].axhline(0, color='black', lw=0.7)
axes[1].set_ylim(-1, 1)
axes[1].set_title('Rolling Correlation (Instability!)', fontweight='bold')
axes[1].set_ylabel('Correlation')
axes[1].legend()

plt.tight_layout()
plt.savefig('correlation.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. Drawdown

### Theory

**Drawdown** measures the loss in portfolio value from the last peak to a subsequent trough:

$$DD_t = \frac{V_t - \max_{s \leq t} V_s}{\max_{s \leq t} V_s}$$

**Maximum Drawdown (MDD):** The largest observed drawdown over the entire period:
$$MDD = \min_t DD_t$$

### Why Drawdown Often Matters More Than Volatility

Volatility is symmetric — it penalises gains and losses equally. Drawdown measures exclusively **negative investor experiences**. MDD is often more relevant psychologically and regulatorily:
- Many institutional mandates have **MDD limits** (e.g. "maximum 15% loss from peak")
- **Hedge fund redemptions:** When a fund is 20% below its peak, institutional investors begin withdrawing capital — a self-reinforcing process
- **High water mark:** Many performance fees are tied to recovery above the last peak

### Calmar Ratio

$$\text{Calmar} = \frac{\text{CAGR}}{|\text{MDD}|}$$

Measures return per unit of drawdown risk. Hedge funds target Calmar > 1.

**Source:** de Prado (2018), Ch. 14; Magdon-Ismail, M. & Atiya, A. (2004). *On the Maximum Drawdown of a Brownian Motion*. Journal of Applied Probability.

In [ ]:
cum_wealth      = (1 + simple_returns).cumprod()
rolling_peak    = cum_wealth.cummax()
drawdown_series = (cum_wealth - rolling_peak) / rolling_peak * 100
mdd             = drawdown_series.min()
calmar          = cagr / abs(mdd / 100)

print(f'Maximum Drawdown : {mdd:.2f}%')
print(f'CAGR             : {cagr*100:.2f}%')
print(f'Calmar Ratio     : {calmar:.3f}')

fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)

axes[0].plot(dates[1:], cum_wealth,   color='#4a90d9', lw=1.8, label='Portfolio Value')
axes[0].plot(dates[1:], rolling_peak, color='#2c3e50', lw=1, ls='--', alpha=0.6, label='Rolling Peak')
axes[0].fill_between(dates[1:], cum_wealth, rolling_peak, alpha=0.15, color='#e74c3c')
axes[0].set_title('Cumulative Portfolio Value & Drawdown', fontweight='bold')
axes[0].set_ylabel('Cumulative Value')
axes[0].legend()

axes[1].fill_between(dates[1:], drawdown_series, 0, color='#e74c3c', alpha=0.5)
axes[1].plot(dates[1:], drawdown_series, color='#c0392b', lw=1)
axes[1].axhline(mdd, color='black', ls='--', lw=1.2, label=f'MDD = {mdd:.2f}%')
axes[1].set_ylabel('Drawdown (%)')
axes[1].set_title(f'Drawdown  |  MDD = {mdd:.2f}%  |  Calmar = {calmar:.3f}', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.savefig('drawdown.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5. Leverage and Margin

### Theory

**Leverage** means deploying more capital than actually available — through borrowing or derivatives. A portfolio with **2x leverage** (200% invested) has double the returns *and* losses:

$$r_{\text{leveraged}} = L \cdot r_{\text{portfolio}} - (L-1) \cdot r_f$$

where $L$ is the leverage factor and $r_f$ the cost of borrowing.

### Margin

**Initial Margin:** Capital that must be posted when entering a position (typically 10–50% of position value for futures).

**Maintenance Margin:** Minimum capital that must remain in the account. If the account falls below this level → **Margin Call**: the broker demands immediate top-up.

**Margin Call:** When the market moves against a leveraged position and equity falls below the maintenance margin, the investor must either:
1. Post additional capital, or
2. Have positions forcibly liquidated

Forced liquidations in stressed markets can trigger **firesale spirals**: many leveraged investors sell simultaneously → prices fall further → more margin calls → more selling. This was a central mechanism of the 2008 financial crisis.

### Leverage and Drawdown — the Asymmetry

Leverage amplifies losses disproportionately due to the **compounding effect**:
- A 50% loss requires +100% gain to recover
- 2x leverage: a 25% market decline means 50% loss of equity
- 10x leverage: a 10% market decline = total loss

**Source:** Hull (2018), Ch. 2; Brunnermeier, M. & Pedersen, L. (2009). *Market Liquidity and Funding Liquidity*. Review of Financial Studies

In [ ]:
r_f_daily       = 0.035 / 252
leverage_levels = [1, 1.5, 2, 3]
colors_lev      = ['#2ecc71', '#f1c40f', '#e67e22', '#e74c3c']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for L, col in zip(leverage_levels, colors_lev):
    r_lev = L * simple_returns - (L - 1) * r_f_daily
    cum   = (1 + r_lev).cumprod()
    axes[0].plot(dates[1:], cum, color=col, lw=1.8, label=f'{L}x Leverage')
    roll_pk = cum.cummax()
    dd_lev  = (cum - roll_pk) / roll_pk * 100
    axes[1].plot(dates[1:], dd_lev, color=col, lw=1.5, alpha=0.85,
                 label=f'{L}x: MDD={dd_lev.min():.1f}%')

axes[0].set_title('Cumulative Value at Different Leverage Levels', fontweight='bold')
axes[0].set_ylabel('Cumulative Value')
axes[0].legend()

axes[1].set_title('Drawdown at Different Leverage Levels', fontweight='bold')
axes[1].set_ylabel('Drawdown (%)')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig('leverage.png', dpi=150, bbox_inches='tight')
plt.show()

print('Leverage Analysis: MDD and Final Value Comparison')
for L, col in zip(leverage_levels, colors_lev):
    r_lev = L * simple_returns - (L - 1) * r_f_daily
    cum   = (1 + r_lev).cumprod()
    mdd_l = ((cum - cum.cummax()) / cum.cummax()).min() * 100
    print(f'  {L}x Leverage: Final Value={cum.iloc[-1]:.3f}, MDD={mdd_l:.1f}%')

---
## 6. Weighted Average Portfolio Returns

### Theory

The **portfolio return** is the **weighted average** of individual asset returns:

$$r_p = \sum_{i=1}^N w_i \cdot r_i = w^T r$$

where $w_i$ are the weights (summing to 1) and $r_i$ are the returns of the individual assets.

### Key Property

This holds **exactly for simple returns**. For log returns it is only an approximation. This is one reason simple returns are preferred for portfolio calculations, even though log returns are better suited for time series analysis.

### Rebalancing and Drift

Over time, weights change due to different returns (**drift**). A portfolio initially split 50/50 across two assets may be 70/30 after a year if one asset has risen strongly. **Rebalancing** periodically resets weights to the target allocation — this systematically enforces "Buy Low / Sell High" and has empirically a small positive return effect (**rebalancing bonus**).

**Source:** Grinold & Kahn (1999), Ch. 2

In [ ]:
assets = ['SAP', 'Allianz', 'Siemens', 'RWE']
vols5  = [0.22, 0.18, 0.20, 0.16]
mus5   = [0.10, 0.08, 0.09, 0.06]
corr4  = np.array([[1,.4,.5,.2],[.4,1,.5,.3],[.5,.5,1,.3],[.2,.3,.3,1]])
d_vols = np.array(vols5) / np.sqrt(252)
d_mus  = np.array(mus5)  / 252
cov4   = np.outer(d_vols, d_vols) * corr4
ret4   = np.random.multivariate_normal(d_mus, cov4, T)
df4    = pd.DataFrame(ret4, columns=assets)
w      = np.array([0.35, 0.25, 0.25, 0.15])

# Buy-and-Hold vs. Monthly Rebalancing
port_bnh = (df4 * w).sum(axis=1)
cum_bnh  = (1 + port_bnh).cumprod()

cum_reb  = pd.Series(np.zeros(T), dtype=float)
wealth   = 1.0
cur_w    = w.copy()
for t in range(T):
    if t % 21 == 0 and t > 0:
        cur_w = w.copy()
    r_t = (df4.iloc[t].values * cur_w).sum()
    wealth *= (1 + r_t)
    cur_w  *= (1 + df4.iloc[t].values)
    cur_w  /= cur_w.sum()
    cum_reb.iloc[t] = wealth

ann_bnh = cum_bnh.iloc[-1] ** (252/T) - 1
ann_reb = cum_reb.iloc[-1] ** (252/T) - 1
print(f'Buy-and-Hold CAGR    : {ann_bnh*100:.2f}%')
print(f'Rebalancing CAGR     : {ann_reb*100:.2f}%')
print(f'Rebalancing Bonus    : {(ann_reb-ann_bnh)*100:.3f}%')

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(dates[1:], cum_bnh, color='#4a90d9', lw=2, label=f'Buy & Hold (CAGR={ann_bnh*100:.1f}%)')
ax.plot(dates[1:], cum_reb, color='#2ecc71', lw=2, ls='--',
        label=f'Monthly Rebalancing (CAGR={ann_reb*100:.1f}%)')
ax.set_title('Portfolio: Buy & Hold vs. Monthly Rebalancing', fontweight='bold')
ax.set_ylabel('Cumulative Value')
ax.legend()
plt.tight_layout()
plt.savefig('rebalancing.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 7. Beta, Alpha and Factor Exposures

### Theory

**Beta** measures the **sensitivity of an asset to a market index** (typically S&P 500, DAX):

$$\beta = \frac{\text{Cov}(r_i, r_m)}{\text{Var}(r_m)} = \rho_{im} \cdot \frac{\sigma_i}{\sigma_m}$$

- $\beta = 1$: Asset moves with the market
- $\beta > 1$: Aggressive — amplifies market moves in both directions (e.g. tech stocks)
- $\beta < 1$: Defensive — dampens market moves (e.g. utilities)
- $\beta < 0$: Inverse movement to market (e.g. gold, some bonds)

**Alpha (Jensen's Alpha)** measures the **risk-adjusted excess return** *not* explained by market risk:

$$\alpha = r_i - [r_f + \beta (r_m - r_f)]$$

Positive alpha means a manager or strategy **delivers more than justified by the market risk taken**. Sustained positive alpha is very rare and the subject of fierce debate (Efficient Market Hypothesis).

### Factor Exposures

Beta is just the simplest factor. Factors are systematic sources of return and risk:
- **Market Factor (MKT):** Beta to the overall market
- **Size (SMB):** Small-cap vs. large-cap premium (Fama-French)
- **Value (HML):** Value vs. growth premium (Fama-French)
- **Momentum (MOM):** Winners tend to continue winning short-term (Carhart)

The factor regression estimates exposures:
$$r_i - r_f = \alpha + \beta_1 \cdot MKT + \beta_2 \cdot SMB + \beta_3 \cdot HML + \beta_4 \cdot MOM + \epsilon$$

**Source:** Fama, E. & French, K. (1993). *Common Risk Factors in the Returns on Stocks and Bonds*. Journal of Financial Economics, 33(1); Carhart, M. (1997). Journal of Finance, 52(1)

In [ ]:
true_beta  = 1.35
true_alpha = 0.0001
r_market   = pd.Series(np.random.normal(0.0003, 0.011, T))
r_specific = pd.Series(np.random.normal(0, 0.008, T))
r_rf_d     = 0.035 / 252
r_stock    = r_rf_d + true_alpha + true_beta * (r_market - r_rf_d) + r_specific

X    = r_market.values - r_rf_d
y    = r_stock.values  - r_rf_d
beta_hat, alpha_hat, r_value, p_value, std_err = stats.linregress(X, y)
r_squared = r_value ** 2

print(f'True   Beta / Alpha     : {true_beta:.3f} / {true_alpha:.6f}')
print(f'Estimated Beta / Alpha  : {beta_hat:.3f} / {alpha_hat:.6f}')
print(f'R²                      : {r_squared:.4f}  ({r_squared*100:.1f}% explained by market)')
print(f'Annualised Alpha        : {alpha_hat*252*100:.3f}%')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter((r_market - r_rf_d) * 100, (r_stock - r_rf_d) * 100,
                alpha=0.2, s=8, color='#4a90d9')
x_fit = np.linspace(X.min(), X.max(), 100)
axes[0].plot(x_fit * 100, (alpha_hat + beta_hat * x_fit) * 100, 'r-', lw=2,
             label=f'β={beta_hat:.3f}, α={alpha_hat*252*100:.3f}% p.a.')
axes[0].set_xlabel('Market Excess Return (%)')
axes[0].set_ylabel('Stock Excess Return (%)')
axes[0].set_title(f'Beta Regression  |  R² = {r_squared:.3f}', fontweight='bold')
axes[0].legend()

roll_beta = pd.Series([
    stats.linregress(r_market.iloc[max(0,i-63):i].values - r_rf_d,
                     r_stock.iloc[max(0,i-63):i].values  - r_rf_d)[0]
    for i in range(63, T)
])
axes[1].plot(roll_beta.values, color='#4a90d9', lw=1.5)
axes[1].axhline(true_beta, color='red',  ls='--', lw=1.5, label=f'True Beta={true_beta}')
axes[1].axhline(1.0,       color='grey', ls=':',  lw=1,   label='Beta=1 (Market)')
axes[1].set_title('Rolling Beta (63-day window)', fontweight='bold')
axes[1].set_ylabel('Beta')
axes[1].legend()

plt.tight_layout()
plt.savefig('beta_alpha.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 8. Fama-French & Carhart Factor Models

### Theory

The **CAPM** explains returns through a single factor (market risk). Fama and French show empirically that two additional factors systematically explain returns:

**Fama-French 3-Factor Model (1993):**
$$r_i - r_f = \alpha + \beta_{MKT} \cdot MKT + \beta_{SMB} \cdot SMB + \beta_{HML} \cdot HML + \epsilon$$

- **SMB (Small Minus Big):** Return of small-cap portfolios minus large-cap portfolios. Historically ~3% p.a. premium.
- **HML (High Minus Low):** Return of value stocks (high book-to-market ratio) minus growth stocks. Historically ~4% p.a. premium.

**Carhart 4-Factor Model (1997):** Adds:
- **MOM (Momentum):** Return of the past 12 months (minus last month) as predictor. Stocks that have performed well tend to continue performing well short-term.

**Fama-French 5-Factor Model (2015):** Adds:
- **RMW (Robust Minus Weak):** Profitability premium
- **CMA (Conservative Minus Aggressive):** Investment premium

### Relevance for Risk Quants

Factor models are the standard tool for **risk attribution**: how much of portfolio risk comes from market risk, size tilt, value tilt etc.? A portfolio with high SMB exposure is vulnerable to small-cap drawdowns — even if the portfolio manager is unaware of it.

**Data source:** Fama-French factors are freely available at https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/data_library.html

**Source:** Fama & French (1993, 2015); Carhart (1997)

In [ ]:
MKT = pd.Series(np.random.normal(0.0003, 0.011, T))
SMB = pd.Series(np.random.normal(0.00012, 0.006, T))
HML = pd.Series(np.random.normal(0.00016, 0.007, T))
MOM = pd.Series(np.random.normal(0.00020, 0.010, T))

true_betas    = {'MKT': 0.95, 'SMB': 0.40, 'HML': 0.20, 'MOM': 0.15}
true_alpha_ff = 0.00005

r_port_ff = (true_alpha_ff
             + true_betas['MKT'] * MKT
             + true_betas['SMB'] * SMB
             + true_betas['HML'] * HML
             + true_betas['MOM'] * MOM
             + pd.Series(np.random.normal(0, 0.005, T)))

factors = pd.DataFrame({'MKT': MKT, 'SMB': SMB, 'HML': HML, 'MOM': MOM})
X_ff    = np.column_stack([np.ones(T), factors.values])
y_ff    = r_port_ff.values
coefs, _, _, _ = np.linalg.lstsq(X_ff, y_ff, rcond=None)
alpha_ff_est = coefs[0]
beta_est     = dict(zip(['MKT', 'SMB', 'HML', 'MOM'], coefs[1:]))
y_hat        = X_ff @ coefs
r2_ff        = 1 - np.sum((y_ff - y_hat)**2) / np.sum((y_ff - y_ff.mean())**2)

print('Carhart 4-Factor Model – Results:')
print(f'  Alpha (ann.)  : {alpha_ff_est*252*100:.3f}%')
print(f'  R²            : {r2_ff:.4f}')
for fname, true_b in true_betas.items():
    print(f'  {fname}: estimated={beta_est[fname]:.3f}, true={true_b:.3f}')

RISK_FREE = 0.035
fig, ax = plt.subplots(figsize=(8, 5))
factor_contributions = {f: beta_est[f] * factors[f].mean() * 252 * 100
                        for f in ['MKT', 'SMB', 'HML', 'MOM']}
factor_contributions['Alpha'] = alpha_ff_est * 252 * 100
cols = ['#e74c3c' if v < 0 else '#2ecc71' for v in factor_contributions.values()]
bars = ax.bar(factor_contributions.keys(), factor_contributions.values(),
              color=cols, edgecolor='white')
ax.bar_label(bars, fmt='%.3f%%', padding=3)
ax.axhline(0, color='black', lw=0.8)
ax.set_ylabel('Contribution to Ann. Return (%)')
ax.set_title('Factor Attribution: Carhart 4-Factor Model', fontweight='bold')
plt.tight_layout()
plt.savefig('factor_model.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Summary – Module 1

| Concept | Core Formula | Regulatory / Practical Relevance |
|---------|-------------|----------------------------------|
| **Log vs. Simple Return** | $r^{log} = \ln(P_t/P_{t-1})$ | Log additive over time; Simple for portfolio weighting |
| **Volatility** | $\sigma_{ann} = \sigma_d \times \sqrt{252}$ | Base input for VaR, margin, option pricing |
| **Correlation** | $\rho = \text{Cov}/(\sigma_i \sigma_j)$ | Diversification; correlation breakdown in crises |
| **Drawdown** | $DD_t = (V_t - \text{Peak}_t)/\text{Peak}_t$ | MDD limits in mandates; Calmar Ratio |
| **Leverage** | $r_{lev} = L \cdot r - (L-1) r_f$ | Margin calls; firesale spirals |
| **Portfolio Return** | $r_p = w^T r$ | Foundation of every portfolio calculation |
| **Beta** | $\beta = \text{Cov}(r_i,r_m)/\text{Var}(r_m)$ | Market risk exposure; CAPM |
| **Alpha** | $\alpha = r_i - [r_f + \beta(r_m-r_f)]$ | Value added beyond market beta |
| **Fama-French** | $r_i - r_f = \alpha + \beta_{MKT} + \beta_{SMB} + \beta_{HML}$ | Risk attribution; factor investing |

---
**→ Continue with Module 2: Portfolio Theory, CAPM, Efficient Frontier**